In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import threading
import os
from typing import Optional, Tuple, Dict, Any

import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

class NanoMOSFETDesigner:
    def __init__(self):
        self.q: tf.Tensor = tf.constant(1.6e-19, dtype=tf.float32)
        self.eps_0: tf.Tensor = tf.constant(8.85e-12, dtype=tf.float32)
        self.eps_ox: float = 3.9
        self.eps_si: float = 11.7
        self.k: tf.Tensor = tf.constant(1.38e-23, dtype=tf.float32)
        self.temperature: tf.Tensor = tf.constant(300.0, dtype=tf.float32)
        self.ni: tf.Tensor = tf.constant(1.5e10, dtype=tf.float32)
        self.phi_ms: float = -0.56

        self.min_Lg: float = 0.1
        self.max_Lg: float = 1.0
        self.max_Lch: float = 5.0

        self.feature_scaler: StandardScaler = StandardScaler()
        self.design_scaler: StandardScaler = StandardScaler()

        self.model: Optional[keras.Model] = None

        self.build_model()

    def build_model(self) -> None:
        inputs = keras.Input(shape=(4,))
        x = layers.Dense(256, activation='tanh')(inputs)
        x = layers.Dense(256, activation='tanh')(x)
        x = layers.Dense(256, activation='tanh')(x)
        outputs = layers.Dense(4)(x)

        self.model = keras.Model(inputs=inputs, outputs=outputs)
        self.model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                         loss='mse')

    def apply_nanoscale_constraints(self, design_params: np.ndarray) -> np.ndarray:
        Lg = np.clip(design_params[:, 0], self.min_Lg, self.max_Lg)
        doping = design_params[:, 1]
        Lch = np.clip(design_params[:, 2], Lg + 0.1, self.max_Lch)
        Vd = design_params[:, 3]
        return np.column_stack([Lg, doping, Lch, Vd])

    def calculate_vd(self, design_params: np.ndarray, input_features: np.ndarray) -> float:
        design_params_tensor = tf.convert_to_tensor(design_params, dtype=tf.float32)
        input_features_tensor = tf.convert_to_tensor(input_features, dtype=tf.float32)

        Lg, Na, Lch, _ = tf.unstack(design_params_tensor, axis=1)
        Id, _, Vtgm, tox = tf.unstack(input_features_tensor, axis=1)

        Lg = tf.clip_by_value(Lg, float(self.min_Lg), float(self.max_Lg))
        Lch = tf.clip_by_value(Lch, Lg + 0.1, float(self.max_Lch))

        Lg_m = Lg * 1e-9
        tox_m = tox * 1e-9
        Na_m = Na * 1e6

        phi_f = (self.k * self.temperature / self.q) * tf.math.log(Na_m / self.ni)
        Cox = self.eps_0 * tf.constant(self.eps_ox, dtype=tf.float32) / tox_m
        Qb = tf.math.sqrt(2.0 * self.q * tf.constant(self.eps_si, dtype=tf.float32) *
                         self.eps_0 * Na_m * phi_f)
        Vth = tf.constant(self.phi_ms, dtype=tf.float32) + 2.0 * phi_f + Qb / Cox

        delta_Vth = 0.1 / (Lg * tox)
        Vth += delta_Vth

        Vd = Vtgm + (Id * Lg_m) / (Cox * (0.7 - Vth) * (Lg/Lch) * 1e-6)

        return float(tf.clip_by_value(Vd, 0.1, 1.2).numpy()[0])

    def predict_design(self, Id: float, SS: float, Vtgm: float, tox: float) -> Dict[str, str]:
        perf_input = np.array([[Id, SS, Vtgm, tox]], dtype=np.float32)
        perf_input_norm = self.feature_scaler.transform(perf_input)

        design_norm = self.model.predict(perf_input_norm, verbose=0)
        design_params = self.design_scaler.inverse_transform(design_norm)

        constrained_params = self.apply_nanoscale_constraints(design_params)
        Lg, doping, Lch, _ = constrained_params[0]

        Vd = self.calculate_vd(constrained_params, perf_input)

        return {
            'Gate length (Lg)': f"{Lg:.3f} nm",
            'Doping concentration': f"{doping:.2e} cm⁻³",
            'Channel length (Lch)': f"{Lch:.3f} nm",
            'Drain voltage (Vd)': f"{Vd:.3f} V",
            'Oxide thickness': f"{tox:.2f} nm"
        }

def load_and_preprocess_data(csv_file: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    try:
        df = pd.read_csv(csv_file)
    except Exception as e:
        raise Exception(f"Error reading CSV file: {e}")

    column_mapping = {
        'Id': ['Id', 'DrainCurrent', 'ID', 'Idrain'],
        'SS': ['SS', 'SubthresholdSwing', 'ss'],
        'Vtgm': ['Vtgm', 'ThresholdVoltage', 'Vth', 'VT'],
        'tox': ['tox', 'OxideThickness', 'TOX'],
        'Lg': ['Lg', 'GateLength', 'Lg_nm'],
        'doping': ['doping', 'DopingConcentration', 'Na', 'Nd'],
        'Lch': ['Lch', 'ChannelLength', 'Lch_nm'],
        'Vd': ['Vd', 'DrainVoltage', 'VDS']
    }

    actual_columns: Dict[str, str] = {}
    for standard_name, possible_names in column_mapping.items():
        for name in possible_names:
            if name in df.columns:
                actual_columns[standard_name] = name
                break

    required_features = ['Id', 'SS', 'Vtgm', 'tox']
    missing_features = [f for f in required_features if f not in actual_columns]

    if missing_features:
        print(f"Warning: Missing columns {missing_features}. Using synthetic data generation.")
        num_samples = len(df)
        X = np.random.randn(num_samples, 4).astype(np.float32)

        Lg = np.random.uniform(0.1, 1.0, num_samples).astype(np.float32)
        doping = (10 ** np.random.uniform(18, 20, num_samples)).astype(np.float32)
        Lch = np.clip(Lg + np.random.uniform(0.1, 4.9, num_samples), None, 5.0).astype(np.float32)
        Vd = np.random.uniform(0.1, 0.8, num_samples).astype(np.float32)
        y = np.column_stack([Lg, doping, Lch, Vd])

        return train_test_split(X, y, test_size=0.2, random_state=42)

    X = df[[actual_columns['Id'], actual_columns['SS'],
            actual_columns['Vtgm'], actual_columns['tox']]].values.astype(np.float32)

    if all(col in actual_columns for col in ['Lg', 'doping', 'Lch', 'Vd']):
        y = df[[actual_columns['Lg'], actual_columns['doping'],
                actual_columns['Lch'], actual_columns['Vd']]].values.astype(np.float32)
        y[:, 0] = np.clip(y[:, 0], 0.1, 1.0)
        y[:, 2] = np.clip(y[:, 2], y[:, 0] + 0.1, 5.0)
    else:
        print("Generating nanoscale-constrained synthetic data...")
        num_samples = len(X)
        Lg = np.random.uniform(0.1, 1.0, num_samples).astype(np.float32)
        doping = (10 ** np.random.uniform(18, 20, num_samples)).astype(np.float32)
        Lch = np.clip(Lg + np.random.uniform(0.1, 4.9, num_samples), None, 5.0).astype(np.float32)
        Vd = np.random.uniform(0.1, 0.8, num_samples).astype(np.float32)
        y = np.column_stack([Lg, doping, Lch, Vd])

    return train_test_split(X, y, test_size=0.2, random_state=42)

class NanoMOSFETDesignerGUI:
    def __init__(self, root: tk.Tk):
        self.root: tk.Tk = root
        self.root.title("NanoMOSFET Designer - Nanoscale MOSFET Design Tool")
        self.root.geometry("1200x800")

        self.root.configure(bg='#2c3e50')

        self.designer: Optional[NanoMOSFETDesigner] = None
        self.is_trained: bool = False
        self.csv_file: Optional[str] = None

        self.main_frame: Optional[ttk.Frame] = None
        self.left_frame: Optional[ttk.Frame] = None
        self.right_frame: Optional[ttk.Frame] = None
        self.inputs: Dict[str, ttk.Entry] = {}
        self.predict_button: Optional[ttk.Button] = None
        self.progress: Optional[ttk.Progressbar] = None
        self.results_text: Optional[tk.Text] = None
        self.figure: Optional[plt.Figure] = None
        self.canvas: Optional[FigureCanvasTkAgg] = None
        self.status_bar: Optional[ttk.Label] = None

        self._setup_styles()
        self._create_menu()
        self._create_main_frame()
        self._create_input_frame()
        self._create_results_frame()
        self._create_plot_frame()

        self.status_bar = ttk.Label(self.root, text="Ready", relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)

    def _setup_styles(self) -> None:
        style = ttk.Style()
        style.theme_use('clam')
        style.configure('Title.TLabel', font=('Helvetica', 16, 'bold'),
                       foreground='white', background='#2c3e50')
        style.configure('Header.TLabel', font=('Helvetica', 12, 'bold'),
                       foreground='#3498db', background='#2c3e50')
        style.configure('TButton', font=('Helvetica', 10), padding=5)
        style.configure('TLabel', background='#2c3e50', foreground='white')
        style.configure('TFrame', background='#2c3e50')
        style.configure('TLabelframe', background='#2c3e50', foreground='white')
        style.configure('TLabelframe.Label', background='#2c3e50', foreground='white')

    def _create_menu(self) -> None:
        menubar = tk.Menu(self.root)
        self.root.config(menu=menubar)

        file_menu = tk.Menu(menubar, tearoff=0)
        menubar.add_cascade(label="File", menu=file_menu)
        file_menu.add_command(label="Load CSV Data", command=self.load_csv_data)
        file_menu.add_command(label="Train Model", command=self.train_model)
        file_menu.add_separator()
        file_menu.add_command(label="Exit", command=self.root.quit)

        help_menu = tk.Menu(menubar, tearoff=0)
        menubar.add_cascade(label="Help", menu=help_menu)
        help_menu.add_command(label="About", command=self._show_about)
        help_menu.add_command(label="User Guide", command=self._show_user_guide)

    def _create_main_frame(self) -> None:
        self.main_frame = ttk.Frame(self.root, padding="10")
        self.main_frame.pack(fill=tk.BOTH, expand=True)

        title_label = ttk.Label(self.main_frame, text="NanoMOSFET Designer", style='Title.TLabel')
        title_label.pack(pady=10)

        desc_text = "Design nanoscale MOSFETs (0.1-1nm gate length) with quantum-aware constraints"
        desc_label = ttk.Label(self.main_frame, text=desc_text, font=('Helvetica', 10), wraplength=800)
        desc_label.pack(pady=5)

        self.left_frame = ttk.Frame(self.main_frame)
        self.left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5)

        self.right_frame = ttk.Frame(self.main_frame)
        self.right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=5)

    def _create_input_frame(self) -> None:
        input_frame = ttk.LabelFrame(self.left_frame, text="Input Parameters", padding="10")
        input_frame.pack(fill=tk.X, pady=5)

        input_fields = [
            ('Drain Current (A)', 'Id', 'e.g., 1e-6'),
            ('Subthreshold Swing (mV/dec)', 'SS', 'e.g., 80'),
            ('Threshold Voltage (V)', 'Vtgm', 'e.g., 0.3'),
            ('Oxide Thickness (nm)', 'tox', 'e.g., 0.5 (0.1-2nm)')
        ]

        for i, (label, key, tooltip) in enumerate(input_fields):
            ttk.Label(input_frame, text=label).grid(row=i, column=0, sticky=tk.W, pady=5, padx=5)
            entry = ttk.Entry(input_frame, width=30)
            entry.grid(row=i, column=1, pady=5, padx=5)
            self.inputs[key] = entry
            self._create_tooltip(entry, tooltip)

        self.predict_button = ttk.Button(input_frame, text="Predict Design",
                                         command=self.predict_design, state=tk.DISABLED)
        self.predict_button.grid(row=len(input_fields), column=0, columnspan=2, pady=20)

        self.progress = ttk.Progressbar(self.left_frame, mode='indeterminate')
        self.progress.pack(fill=tk.X, pady=5)

    def _create_results_frame(self) -> None:
        results_frame = ttk.LabelFrame(self.left_frame, text="Design Results", padding="10")
        results_frame.pack(fill=tk.X, pady=5)

        self.results_text = tk.Text(results_frame, height=12, width=40, font=('Courier', 10),
                                   bg='#34495e', fg='#2ecc71', relief=tk.FLAT)
        self.results_text.pack(fill=tk.BOTH, expand=True)

        copy_button = ttk.Button(results_frame, text="Copy Results", command=self.copy_results)
        copy_button.pack(pady=5)

    def _create_plot_frame(self) -> None:
        plot_frame = ttk.LabelFrame(self.right_frame, text="Training History", padding="10")
        plot_frame.pack(fill=tk.BOTH, expand=True, pady=5)

        self.figure = plt.Figure(figsize=(5, 4), dpi=100, facecolor='#2c3e50')
        self.canvas = FigureCanvasTkAgg(self.figure, master=plot_frame)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def _create_tooltip(self, widget: ttk.Entry, text: str) -> None:
        tooltip_window: Optional[tk.Toplevel] = None

        def show_tooltip(event: tk.Event) -> None:
            nonlocal tooltip_window
            if tooltip_window:
                return
            tooltip_window = tk.Toplevel()
            tooltip_window.wm_overrideredirect(True)
            tooltip_window.wm_geometry(f"+{event.x_root+10}+{event.y_root+10}")
            label = tk.Label(tooltip_window, text=text, background="#ffffe0",
                            relief=tk.SOLID, borderwidth=1)
            label.pack()

        def hide_tooltip(event: tk.Event) -> None:
            nonlocal tooltip_window
            if tooltip_window:
                tooltip_window.destroy()
                tooltip_window = None

        widget.bind('<Enter>', show_tooltip)
        widget.bind('<Leave>', hide_tooltip)

    def load_csv_data(self) -> None:
        filename = filedialog.askopenfilename(
            title="Select MOSFET Data CSV",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )

        if filename:
            self.csv_file = filename
            if self.status_bar:
                self.status_bar.config(text=f"Loaded: {os.path.basename(filename)}")
            messagebox.showinfo("Success", f"Data loaded from {os.path.basename(filename)}")

    def train_model(self) -> None:
        if not self.csv_file:
            messagebox.showwarning("No Data", "Please load CSV data first")
            return

        if self.progress:
            self.progress.start()
        if self.predict_button:
            self.predict_button.config(state=tk.DISABLED)
        if self.status_bar:
            self.status_bar.config(text="Training model...")

        thread = threading.Thread(target=self._train_model_thread)
        thread.daemon = True
        thread.start()

    def _train_model_thread(self) -> None:
        try:
            X_train, X_test, y_train, y_test = load_and_preprocess_data(self.csv_file)

            self.designer = NanoMOSFETDesigner()
            self.designer.feature_scaler.fit(X_train)
            self.designer.design_scaler.fit(y_train)

            history = self.designer.model.fit(
                self.designer.feature_scaler.transform(X_train),
                self.designer.design_scaler.transform(y_train),
                epochs=200,
                batch_size=32,
                validation_split=0.2,
                verbose=0
            )

            self.is_trained = True
            self.root.after(0, self._training_complete, history)

        except Exception as e:
            self.root.after(0, self._training_error, str(e))

    def _training_complete(self, history: Any) -> None:
        if self.progress:
            self.progress.stop()
        if self.predict_button:
            self.predict_button.config(state=tk.NORMAL)
        if self.status_bar:
            self.status_bar.config(text="Model trained successfully!")
        messagebox.showinfo("Success", "Model training completed!")
        self._plot_training_history(history)

    def _training_error(self, error: str) -> None:
        if self.progress:
            self.progress.stop()
        if self.predict_button:
            self.predict_button.config(state=tk.DISABLED)
        if self.status_bar:
            self.status_bar.config(text="Training failed")
        messagebox.showerror("Error", f"Training failed: {error}")

    def _plot_training_history(self, history: Any) -> None:
        if not self.figure or not self.canvas:
            return

        self.figure.clear()
        ax = self.figure.add_subplot(111)
        ax.plot(history.history['loss'], label='Training Loss', linewidth=2, color='#3498db')
        ax.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='#e74c3c')
        ax.set_title('Model Training History', fontsize=12, fontweight='bold', color='white')
        ax.set_xlabel('Epoch', color='white')
        ax.set_ylabel('RMSE Loss', color='white')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_yscale('log')
        ax.tick_params(colors='white')
        ax.set_facecolor('#34495e')
        self.figure.patch.set_facecolor('#2c3e50')
        self.canvas.draw()

    def predict_design(self) -> None:
        if not self.is_trained:
            messagebox.showwarning("Model Not Trained", "Please train the model first")
            return

        try:
            Id = float(self.inputs['Id'].get())
            SS = float(self.inputs['SS'].get())
            Vtgm = float(self.inputs['Vtgm'].get())
            tox = float(self.inputs['tox'].get())

            if Id <= 0 or SS <= 0 or tox <= 0:
                raise ValueError("All values must be positive")

            if tox < 0.1 or tox > 2:
                messagebox.showwarning("Warning", "Oxide thickness outside typical nanoscale range (0.1-2nm)")

            if self.status_bar:
                self.status_bar.config(text="Predicting design...")

            result = self.designer.predict_design(Id, SS, Vtgm, tox)
            self._display_results(result)

            if self.status_bar:
                self.status_bar.config(text="Prediction completed")

        except ValueError as e:
            messagebox.showerror("Invalid Input", f"Please enter valid numbers\nError: {str(e)}")
        except Exception as e:
            messagebox.showerror("Error", f"Prediction failed: {str(e)}")

    def _display_results(self, result: Dict[str, str]) -> None:
        if not self.results_text:
            return

        self.results_text.delete(1.0, tk.END)

        results_str = "=" * 40 + "\n"
        results_str += "NANOSCALE MOSFET DESIGN\n"
        results_str += "=" * 40 + "\n\n"

        for param, value in result.items():
            results_str += f"{param:25}: {value}\n"

        results_str += "\n" + "=" * 40 + "\n"
        results_str += "Quantum-aware design with\n"
        results_str += "Lg: 0.1-1nm constraints applied\n"

        self.results_text.insert(1.0, results_str)

    def copy_results(self) -> None:
        if self.results_text:
            results = self.results_text.get(1.0, tk.END)
            self.root.clipboard_clear()
            self.root.clipboard_append(results)
            if self.status_bar:
                self.status_bar.config(text="Results copied to clipboard")

    @staticmethod
    def _show_about() -> None:
        about_text = """NanoMOSFET Designer v1.0

A machine learning tool for designing nanoscale MOSFETs with:
- Gate lengths: 0.1-1nm (atomic scale)
- Channel lengths: ≤5nm
- Quantum-aware threshold voltage calculation
- Physics-based drain voltage estimation

Features:
- Neural network-based inverse design
- Nanoscale constraint enforcement
- Interactive GUI
- Training visualization

Developed for nanoscale semiconductor device design"""
        messagebox.showinfo("About", about_text)

    @staticmethod
    def _show_user_guide() -> None:
        guide_text = """User Guide:

1. Load Data:
   - Click File → Load CSV Data
   - Select CSV file with MOSFET data
   - Required columns: Id, SS, Vtgm, tox
   - Optional columns: Lg, doping, Lch, Vd

2. Train Model:
   - Click File → Train Model
   - Wait for training completion (progress bar)
   - Training history will be displayed

3. Make Predictions:
   - Enter input parameters
   - Click "Predict Design"
   - Results show Lg, doping, Lch, Vd

4. Nanoscale Constraints:
   - Lg: 0.1-1nm automatically enforced
   - Lch: ≥ Lg+0.1nm, ≤5nm
   - Vd: Limited to 0.1-1.2V"""
        messagebox.showinfo("User Guide", guide_text)

def main() -> None:
    root = tk.Tk()
    NanoMOSFETDesignerGUI(root)
    root.mainloop()

if __name__ == "__main__":
    main()